# Этап 1. Знакомство с данными и аудит качества

## Подключение к БД

In [5]:
import sys
from pathlib import Path

# добавление папки src в места поиска модулей
sys.path.append(str(Path.cwd().parent / 'src'))

from db import query

## Проверка подключения к базе и вывод названий всех таблиц в БД

In [6]:
query('SELECT name FROM sqlite_master WHERE type = "table"')

,name
0,users
1,devices
2,app_releases
3,screens
4,sessions
5,screen_views
6,events
7,orders
8,ab_assignments


---

# <u>Просмотр всех таблиц и столбцов в них для планирования анализа данных</u>


In [7]:
query('SELECT'
      ' sm.name AS table_name, '
      ' t_info.name AS column_name, '
      ' t_info.type AS column_type '
      'FROM sqlite_master sm '
      'JOIN pragma_table_info(sm.name) t_info '
      'WHERE sm.type = "table" '
      'ORDER BY sm.name, t_info.cid;')

,table_name,column_name,column_type
0,ab_assignments,user_id,INTEGER
1,ab_assignments,experiment,TEXT
2,ab_assignments,variant,TEXT
3,ab_assignments,assigned_at,TEXT
4,app_releases,platform,TEXT
5,app_releases,app_version,TEXT
6,app_releases,released_at,TEXT
7,app_releases,notes,TEXT
8,devices,device_id,TEXT
9,devices,user_id,INTEGER


---

## <u>1.1 Анализ дат на достоверность</u>

In [32]:
query('''
SELECT 'sessions start' AS tbl, COUNT(*) AS rows,
MIN(started_at) AS min_date, MAX(started_at) AS max_date
FROM sessions
UNION ALL
SELECT 'sessions end', COUNT(*),
MIN(ended_at), MAX(ended_at)
FROM sessions
UNION ALL
SELECT 'orders create', COUNT(*),
MIN(created_at), MAX(created_at)
FROM orders
UNION ALL
SELECT 'screen_views enter', COUNT(*),
MIN(entered_at), MAX(entered_at)
FROM screen_views
UNION ALL
SELECT 'users registrate', COUNT(*),
MIN(registered_at), MAX(registered_at)
FROM users
UNION ALL
SELECT 'devices first seen', COUNT(*),
MIN(first_seen_at), MAX(first_seen_at)
FROM devices
UNION ALL
SELECT 'app_releases release', COUNT(*),
MIN(released_at), MAX(released_at)
FROM app_releases
UNION ALL
SELECT 'ab_assignments assign', COUNT(*),
MIN(assigned_at), MAX(assigned_at)
FROM ab_assignments
UNION ALL
SELECT 'events client', COUNT(*),
MIN(client_ts), MAX(client_ts)
FROM events
UNION ALL
SELECT 'events server', COUNT(*),
MIN(server_ts), MAX(server_ts)
FROM events
UNION ALL
SELECT 'screens info', COUNT(*),
NULL, NULL
FROM screens;
''')

,tbl,rows,min_date,max_date
0,sessions start,49191,2026-05-31 21:40:05,2026-08-31 18:48:39
1,sessions end,49191,2026-05-31 21:40:55,2026-08-31 18:56:54
2,orders create,21954,2026-05-31 23:20:55,2026-08-31 18:50:38
3,screen_views enter,286078,2026-05-31 21:40:05,2026-08-31 18:56:48
4,users registrate,9006,2025-01-01 07:08:27,2026-08-30 18:40:34
5,devices first seen,9006,2025-01-01 07:08:27,2026-08-30 18:40:34
6,app_releases release,9,2026-05-20,2026-08-13
7,ab_assignments assign,9031,2026-06-30 21:01:00,2026-08-31 18:36:19
8,events client,270273,2025-05-04 08:33:10,2027-07-11 03:44:33
9,events server,270273,2026-05-31 21:40:05,2026-08-31 23:39:16


#### По данной таблице можно увидеть, что:
1. Самое позднее время во всех таблицах это 18:56 и оно почти везде одинаковое, значит можно сделать вывод, что в БД время отличается от Узбекистана (UTC+5), если исправить, то время увеличится на 5 часов - это и будет граница периода по ТЗ
2. По объему данных:
    1)  ~ 9 000 пользователей - верно
    2)  ~ 49 000 сессий - верно
    3)  ~ 286 000 просмотров экранов - верно
    4)  ~ 270 000 событий - верно
    5)  ~ 22 000 заказов - верно
3. У таблицы  app_releases формат даты отличен от остальных. Требует внимания!
4. В таблицах users и devices полное совпадение минимумов и максимумов. То есть 1 пользователь = 1 устройство. Нужно потом разобраться почему так? и почему момент регистрации совпадает с первым подключением устройства до секунды?
5. events: Почему у события на стороне Клиента не совпадают с датами событий на стороне Сервера? Почему события сервера вписываются в диапазон по ТЗ, а события на стороне Клиента имеют больший разброс по датам (количество строк у них одинаковое) и максимум уходит в будущее? Может ли проблема быть на стороне Клиента?

### Время на стороне Клиента ненадежно, поэтому для дальнейшего анализа буду использовать время Сервера. Со временем на стороне Клиента разберусь в следующих шагах

---


## <u>1.2 Поиск пропусков в данных по ключевым полям</u>

#### Вывод всех полей, где есть пропуски и их процентное соотношение

In [15]:
query('''
WITH nulls AS (
      SELECT 'users' AS table_name, 'city' AS column_name,
             ROUND(AVG(city IS NULL) * 100, 1) AS null_pct
      FROM users
      UNION ALL
      SELECT 'users', 'registered_at',
             ROUND(AVG(registered_at IS NULL) * 100, 1)
      FROM users
      UNION ALL
      SELECT 'users', 'birth_year',
             ROUND(AVG(birth_year IS NULL) * 100, 1)
      FROM users
      UNION ALL
      SELECT 'users', 'acquisition_channel',
             ROUND(AVG(acquisition_channel IS NULL) * 100, 1)
      FROM users
      UNION ALL
      SELECT 'devices', 'user_id',
             ROUND(AVG(user_id IS NULL) * 100, 1)
      FROM devices
      UNION ALL
      SELECT 'devices', 'platform',
             ROUND(AVG(platform IS NULL) * 100, 1)
      FROM devices
      UNION ALL
      SELECT 'devices', 'device_model',
             ROUND(AVG(device_model IS NULL) * 100, 1)
      FROM devices
      UNION ALL
      SELECT 'devices', 'os_version',
             ROUND(AVG(os_version IS NULL) * 100, 1)
      FROM devices
      UNION ALL
      SELECT 'sessions', 'user_id',
             ROUND(AVG(user_id IS NULL) * 100, 1)
      FROM sessions
      UNION ALL
      SELECT 'sessions', 'device_id',
             ROUND(AVG(device_id IS NULL) * 100, 1)
      FROM sessions
      UNION ALL
      SELECT 'sessions', 'started_at',
             ROUND(AVG(started_at IS NULL) * 100, 1)
      FROM sessions
      UNION ALL
      SELECT 'sessions', 'ended_at',
             ROUND(AVG(ended_at IS NULL) * 100, 1)
      FROM sessions
      UNION ALL
      SELECT 'sessions', 'app_version',
             ROUND(AVG(app_version IS NULL) * 100, 1)
      FROM sessions
      UNION ALL
      SELECT 'sessions', 'network_type',
             ROUND(AVG(network_type IS NULL) * 100, 1)
      FROM sessions
      UNION ALL
      SELECT 'screen_views', 'session_id',
             ROUND(AVG(session_id IS NULL) * 100, 1)
      FROM screen_views
      UNION ALL
      SELECT 'screen_views', 'screen_name',
             ROUND(AVG(screen_name IS NULL) * 100, 1)
      FROM screen_views
      UNION ALL
      SELECT 'screen_views', 'entered_at',
             ROUND(AVG(entered_at IS NULL) * 100, 1)
      FROM screen_views
      UNION ALL
      SELECT 'screen_views', 'duration_ms',
             ROUND(AVG(duration_ms IS NULL) * 100, 1)
      FROM screen_views
      UNION ALL
      SELECT 'events', 'session_id',
             ROUND(AVG(session_id IS NULL) * 100, 1)
      FROM events
      UNION ALL
      SELECT 'events', 'user_id',
             ROUND(AVG(user_id IS NULL) * 100, 1)
      FROM events
      UNION ALL
      SELECT 'events', 'client_ts',
             ROUND(AVG(client_ts IS NULL) * 100, 1)
      FROM events
      UNION ALL
      SELECT 'events', 'server_ts',
             ROUND(AVG(server_ts IS NULL) * 100, 1)
      FROM events
      UNION ALL
      SELECT 'events', 'event_name',
             ROUND(AVG(event_name IS NULL) * 100, 1)
      FROM events
      UNION ALL
      SELECT 'events', 'screen_name',
             ROUND(AVG(screen_name IS NULL) * 100, 1)
      FROM events
      UNION ALL
      SELECT 'events', 'event_params',
             ROUND(AVG(event_params IS NULL) * 100, 1)
      FROM events
      UNION ALL
      SELECT 'orders', 'service',
             ROUND(AVG(service IS NULL) * 100, 1)
      FROM orders
      UNION ALL
      SELECT 'orders', 'tariff',
             ROUND(AVG(tariff IS NULL) * 100, 1)
      FROM orders
      UNION ALL
      SELECT 'orders', 'status',
             ROUND(AVG(status IS NULL) * 100, 1)
      FROM orders
      UNION ALL
      SELECT 'orders', 'price_uzs',
             ROUND(AVG(price_uzs IS NULL) * 100, 1)
      FROM orders
      UNION ALL
      SELECT 'orders', 'wait_time_sec',
             ROUND(AVG(wait_time_sec IS NULL) * 100, 1)
      FROM orders
      UNION ALL
      SELECT 'orders', 'rating',
             ROUND(AVG(rating IS NULL) * 100, 1)
      FROM orders
      UNION ALL
      SELECT 'orders', 'promo_applied',
             ROUND(AVG(promo_applied IS NULL) * 100, 1)
      FROM orders
      UNION ALL
      SELECT 'ab_assignments', 'variant',
             ROUND(AVG(variant IS NULL) * 100, 1)
      FROM ab_assignments
      UNION ALL
      SELECT 'ab_assignments', 'assigned_at',
             ROUND(AVG(assigned_at IS NULL) * 100, 1)
      FROM ab_assignments
      UNION ALL
      SELECT 'app_releases', 'released_at',
             ROUND(AVG(released_at IS NULL) * 100, 1)
      FROM app_releases
      UNION ALL
      SELECT 'app_releases', 'notes',
             ROUND(AVG(notes IS NULL) * 100, 1)
      FROM app_releases
      UNION ALL
      SELECT 'screens', 'service',
             ROUND(AVG(service IS NULL) * 100, 1)
      FROM screens
      UNION ALL
      SELECT 'screens', 'description',
             ROUND(AVG(description IS NULL) * 100, 1)
      FROM screens
)
SELECT table_name, column_name, null_pct
FROM nulls
WHERE null_pct > 0
ORDER BY table_name, null_pct DESC;
''')

,table_name,column_name,null_pct
0,events,event_params,21.4
1,events,screen_name,19.5
2,events,user_id,3.2
3,orders,rating,64.0
4,orders,tariff,22.7
5,orders,wait_time_sec,22.7
6,orders,price_uzs,19.0
7,sessions,ended_at,0.9
8,users,birth_year,4.8


### Результат: из 38 проверенных полей пропуски есть только в 9
---

### <u>Анализ пропусков по таблице `events`</u>

In [8]:
query('''
WITH events_params_null AS (
      SELECT ev.event_name AS event_name,
            COUNT(*) AS event_count,
            ROUND(AVG(ev.event_params IS NULL) * 100, 1) AS params_null_pct
      FROM events ev
      GROUP BY ev.event_name
)
SELECT event_name, event_count, params_null_pct
FROM events_params_null
WHERE params_null_pct > 0
ORDER BY params_null_pct DESC;
''')

,event_name,event_count,params_null_pct
0,app_background,49315,100.0
1,favorite_address_add,262,100.0
2,favorite_address_click,4182,100.0
3,no_cars_found,1362,100.0
4,onboarding_skip,2653,100.0


#### Пропуски только в 5 видах событий и у каждого из них в 100 % случаев, у остальных - 0 %.
делаю вывод, что это норма.

In [12]:
query('''
WITH events_screen_null AS (
      SELECT ev.event_name AS event_name,
            COUNT(*) AS event_count,
            ROUND(AVG(ev.screen_name IS NULL) * 100, 1) AS screen_null_pct
      FROM events ev
      GROUP BY ev.event_name
)
SELECT event_name, event_count, screen_null_pct
FROM events_screen_null
WHERE screen_null_pct > 0
ORDER BY screen_null_pct DESC;
''')

,event_name,event_count,screen_null_pct
0,app_background,49315,100.0
1,push_open,3493,100.0


#### Пропуски только у системных событий:
   1. `app_background` - сворачивание приложения
   2. `push_open` - открытие из push уведомления
происходят вне какого-либо определенного экрана. Норма
---

### <u>Анализ пропусков по таблице `orders`</u>

In [9]:
query('''
SELECT o.service,
       o.status,
       COUNT(*) AS orders_cnt,
       ROUND(AVG(o.tariff IS NULL) * 100, 1)        AS tariff_null_pct,
       ROUND(AVG(o.wait_time_sec IS NULL) * 100, 1) AS wait_null_pct,
       ROUND(AVG(o.rating IS NULL) * 100, 1)        AS rating_null_pct
FROM orders AS o
GROUP BY o.service, o.status
ORDER BY o.service, o.status;
''')

,service,status,orders_cnt,tariff_null_pct,wait_null_pct,rating_null_pct
0,food,cancelled_restaurant,136,100.0,100.0,100.0
1,food,cancelled_user,427,100.0,100.0,100.0
2,food,completed,3045,100.0,100.0,100.0
3,scooter,completed,1377,100.0,100.0,100.0
4,taxi,cancelled_driver,513,0.0,0.0,100.0
5,taxi,cancelled_user,1749,0.0,0.0,100.0
6,taxi,completed,13359,0.0,0.0,40.9
7,taxi,no_cars,1348,0.0,0.0,100.0


#### По этой таблице видно, что пропуски объяснимы:
1. у еды и самокатов нет тарифа, время ожидания и оценки поездки
2. у такси поля тариф и время ожидания заполнены на 100 %
3. у отменных заказов или `no_cars` нет рейтинга - норма
4. среди выполненных заказов 40,9% неоцененных поездок - похоже на правду
---

### <u>Анализ оставшихся пропусков</u>

In [14]:
query('''
SELECT
      (SELECT ROUND(AVG(birth_year IS NULL) * 100, 1) FROM users)  AS user_birth_year_null,
      (SELECT ROUND(AVG(ended_at IS NULL) * 100, 1) FROM sessions) AS sessions_ended_at_null,
      (SELECT ROUND(AVG(user_id IS NULL) * 100, 1) FROM events)    AS events_user_id_null,
      (SELECT ROUND(AVG(price_uzs IS NULL) * 100, 1) FROM orders)  AS orders_price_uzs_null,
      (SELECT ROUND(AVG(price_uzs IS NULL) * 100, 1) FROM orders WHERE status = 'completed') AS completed_price_uzs_null;
''')

,user_birth_year_null,sessions_ended_at_null,events_user_id_null,orders_price_uzs_null,completed_price_uzs_null
0,4.8,0.9,3.2,19.0,0.0


#### По пропускам из таблицы видно, что:
   1. Есть 4,8 % пропусков в поле `users.birth_year`. Это значит, что пользователь не указал дату рождения, значит нужно исключать из подсчета возраста;
   2. Есть 0,9 % пропусков в `sessions.ended_at` - Сигнал, что сессия оборвалась. Проверить на этапе стабильности;
   3. Есть 3,2 % пропусков в `events.user_id` - это нормально, так как события до входа в аккаунт анонимны;
   4. Есть 19 % пропусков в `orders.price_uzs` - это нормально, так как у выполненных заказов `completed_price_uzs` пропусков 0 % - цены нет только у незавершенных заказов
---

## По результатам шага 1.2 обнаружен 1 сигнал в поле `sessions.ended_at` - обрыв сессии

#### Рассмотреть на этапе стабильности
---

## <u>1.3 Проверка связности таблиц</u>

In [30]:
query('''
-- использую LEFT JOIN, так как он сохраняет все строки левой таблицы,
-- если пары нет - будет NULL
SELECT
      -- заказы без сессии:
      (SELECT COUNT(*)
      FROM orders o
      LEFT JOIN sessions s ON s.session_id = o.session_id
      WHERE s.session_id IS NULL) AS orders_without_sessions,
      -- сессии без пользователя:
      (SELECT COUNT(*)
      FROM sessions s
      LEFT JOIN users u ON u.user_id = s.user_id
      WHERE u.user_id IS NULL) AS sessions_without_user,
      -- события без сессии:
      (SELECT COUNT(*)
      FROM events e
      LEFT JOIN sessions s ON s.session_id = e.session_id
      WHERE s.session_id IS NULL) AS events_without_sessions,
      -- просмотры экранов без сессии:
      (SELECT COUNT(*)
      FROM screen_views sv
      LEFT JOIN sessions s ON s.session_id = sv.session_id
      WHERE s.session_id IS NULL) AS screen_views_without_sessions,
      -- пользователи без устройств
      (SELECT COUNT(*)
      FROM users u
      LEFT JOIN devices d ON d.user_id = u.user_id
      WHERE d.user_id IS NULL) AS users_without_devices;
''')

,orders_without_sessions,users_without_sessions,events_without_sessions,screen_views_without_sessions,users_without_devices
0,0,0,0,0,0


#### Связи между таблицами проверены:
1. заказы -- сессии
2. сессии -- пользователи
3. события -- сессии
4. просмотры экранов -- сессии
5. пользователи -- устройства

#### Сирот нет ни в одной паре таблиц. + выяснила, что 1 пользователь = 1 устройство
---